# title: "SIMPlex Mouse Brain — Visium spatial transcriptomics and cell type deconvolution"
subtitle: "Manuscript figures: Fig. 1f, Extended Data Fig. 1c–e"
author: "Mengxiao He"
output: html_document



In [ ]:
knitr::opts_chunk$set(echo = TRUE)

In [ ]:
# DATA_ROOT, HDF5 library, palettes — single source of truth for paths.
source(here::here("config.R"))



## Load libraries

Load libraries



In [ ]:
suppressMessages(require(semla))
suppressMessages(require(patchwork))
suppressMessages(require(ggplot2))
suppressMessages(require(SeuratDisk))
suppressMessages(require(magick))
suppressMessages(require(scCustomize))
suppressMessages(require(tidyverse))



## Load data



In [ ]:
# Load BC Visium samples
visium.dir <- paste0(SPACERANGER, "/mouse_brain")

samples <- list.files(visium.dir, recursive = TRUE, full.names = TRUE, pattern = 'filtered_feature_bc_matrix.h5')

imgs <- list.files(visium.dir, recursive = TRUE, full.names = TRUE, pattern = 'tissue_hires_image.png')

spotfiles <- list.files(visium.dir, recursive = TRUE, full.names = TRUE, pattern = 'tissue_positions.csv')

json <- list.files(visium.dir, recursive = TRUE, full.names = TRUE, pattern = 'scalefactors_json.json')

section.name <- samples
section.name <- gsub(paste0(visium.dir, "/"),"", gsub("/filtered_feature_bc_matrix.h5", "", section.name))

infoTable <- tibble(section.name, samples, imgs, spotfiles, json, # Add required columns
                    sample_id = c("MB 1.1", "MB 1.2")) # Add additional column

se <- ReadVisiumData(infoTable, verbose = FALSE)



## Filter by number of unique genes



In [ ]:
# Filter by spots containing more than 400 genes and genes with more than 300 UMIs as well as at least 5 spots
nCount.UMI <- Matrix::rowSums(GetAssayData(se))
spatial.genes <- Matrix::rowSums(GetAssayData(se) > 0)
gene.data <- data.frame(nCounts = nCount.UMI, spatial.genes = spatial.genes)
min_expr <- gene.data$nCounts >= 300
min_obs <- gene.data$spatial.genes >= 5
keep.genes <- rownames(se)[min_expr & min_obs]
se_filtered <- SubsetSTData(se, features = keep.genes, expression = nFeature_Spatial > 400)
rm(se)



## Filtered feature plot


In [ ]:
MapFeaturesSummary(se_filtered, features = "nFeature_Spatial", ncol = 2, subplot_type = "histogram")



## Transform spatial image



In [ ]:
se_filtered <- LoadImages(se_filtered, image_height = 2000, verbose = FALSE)

transforms <- generate_rigid_transform(sampleID = 2, angle = -90)
se_filtered <- RigidTransformImages(se_filtered, transforms = transforms, verbose = FALSE)



## Normalize and find variable features of spatial data



In [ ]:
# Normalize data and find variable features for Visium data
se_filtered <- se_filtered |>
  NormalizeData(verbose = FALSE) |>
  FindVariableFeatures(nfeatures = 10000, verbose = FALSE)



## Import SIMPlex snRNAseq data and original scRNAseq data
# With label transfer annotations


In [ ]:
#SIMPlex label transfer
snMB <- readRDS(paste0(SN_RDS, "/mouse_brain/snRNA.rds"))



## Normalize data and find variable features



In [ ]:
# Rerun FindVariableFeatures to increase the number before cell type deconvolution
snMB <- snMB |> 
  FindVariableFeatures(nfeatures = 10000, verbose = FALSE)



## Deconvolution with NNLS using SIMPlex (labeltransfer)



In [ ]:
DefaultAssay(se_filtered) <- "Spatial"

se_filtered_sub <- RunNNLS(object = se_filtered, 
                            singlecell_object = snMB, 
                            groups = "subclass", nCells_per_group = 8000)
se_filtered_tax4 <- RunNNLS(object = se_filtered, 
                            singlecell_object = snMB, 
                            groups = "tax4", nCells_per_group = 4000)

rm(snMB_linnarsson_sub, snMB_linnarsson_tax4)



## Plot cell types SIMPlex from Linnarsson subclass



In [ ]:
DefaultAssay(se_filtered_sub) <- "celltypeprops"

selected_celltypes <- c("Astrocyte", "Ependymal", "Immune", "Neurons", "Oligos", "Ttr", "Vascular")

MapFeatures(se_filtered_sub,
            pt_size = 2,
            section_number = 2,
            ncol = 3,
            image_use = "transformed",
            features = c("Astrocyte", "Neurons", "Oligos"),
            colors = RColorBrewer::brewer.pal(n = 9, name = "Spectral") |> rev(),
            scale_alpha = TRUE,
            crop_area = c(0.18, 0.48, 0.82, 0.88)) &
  theme(legend.position = "right", legend.margin = margin(b = 50),
        legend.text = element_text(angle = 0),
        plot.title = element_blank())



## Multiple cell types



In [ ]:
# Plot multiple features
multi_plot_sub <- MapMultipleFeatures(se_filtered_sub, 
                                      section_number = 2, 
                                      image_use = "transformed", 
                                      pt_size = 1, 
                                      max_cutoff = 0.99, 
                                      colors = c("#F8766D", "#CD9600", "#7CAE00", "#00BE67", "#00A9FF", "#C77CFF","#FF61CC"),
                                      features = selected_celltypes[1:7], 
                                      crop_area = c(0.18, 0.48, 0.82, 0.88)) & 
  theme(plot.title = element_blank())



## Plot cell types SIMPlex from Linnarsson taxomony 4



In [ ]:
DefaultAssay(se_filtered_tax4) <- "celltypeprops"

selected_celltypes <- c("Astrocytes", "Choroid epithelial cells", "Dentate gyrus granule neurons", "Di- and mesencephalon excitatory neurons", "Di- and mesencephalon inhibitory neurons", "Ependymal cells", "Hindbrain neurons", "Microglia", "Non-glutamatergic neuroblasts", "Oligodendrocyte precursor cells", "Oligodendrocytes", "Peptidergic neurons", "Pericytes", "Telencephalon inhibitory interneurons", "Telencephalon projecting excitatory neurons", "Telencephalon projecting inhibitory neurons", "Vascular and leptomeningeal cells", "Vascular endothelial cells", "Vascular smooth muscle cells")

se_filtered_tax4 <- LoadImages(se_filtered_tax4, verbose = FALSE)

MapFeatures(se_filtered_tax4,
            pt_size = 1.5,
            section_number = 2,
            ncol = 4,
            image_use = "transformed",
            features = c("Astrocytes",
                         "Dentate gyrus granule neurons",
                         "Di- and mesencephalon excitatory neurons",
                         "Di- and mesencephalon inhibitory neurons",
                         "Oligodendrocytes",
                         "Peptidergic neurons",
                         "Telencephalon projecting excitatory neurons",
                         "Telencephalon projecting inhibitory neurons"),
            colors = RColorBrewer::brewer.pal(n = 9, name = "Spectral") |> rev(),
            scale_alpha = TRUE,
            crop_area = c(0.18, 0.48, 0.82, 0.88)) &
  theme(legend.position = "right", legend.margin = margin(b = 50),
        legend.text = element_text(angle = 0),
        plot.title = element_blank())



## Multiple cell types (all)



In [ ]:
# Plot multiple features
selected_celltypes <- c("Astrocytes", "Choroid epithelial cells", "Dentate gyrus granule neurons", "Di- and mesencephalon excitatory neurons", "Di- and mesencephalon inhibitory neurons", "Ependymal cells", "Hindbrain neurons", "Non-glutamatergic neuroblasts", "Oligodendrocytes", "Peptidergic neurons", 
                        "Pericytes", "Telencephalon inhibitory interneurons", "Telencephalon projecting excitatory neurons", "Telencephalon projecting inhibitory neurons")
multi_plot_tax4 <- MapMultipleFeatures(se_filtered_tax4, 
                                       section_number = 2,
                                       image_use = "transformed", 
                                       pt_size = 1, max_cutoff = 0.99,
                                       colors = c("#F8766D", "#D89000", "#C09B00", "#A3A500", "#7CAE00", "#39B600", "#00BB4E", "#00C1A3", "#00BAE0", "#00B0F6", "#35A2FF", 
                                                  "#9590FF", "#C77CFF", "#E76BF3"),
                                       features = selected_celltypes[1:14],
                                       crop_area = c(0.18, 0.48, 0.82, 0.88)) & 
  theme(plot.title = element_blank())



## Deconvolution with NNLS using labelstransfered data (Allen)



In [ ]:
DefaultAssay(se_filtered) <- "Spatial"

se_filtered_allen_simp <- RunNNLS(object = se_filtered, 
                            singlecell_object = snMB, 
                            groups = "allen_cortex", nCells_per_group = 2000)

rm(snMB_SIMP_allen)




## Plot cell types using SIMPlex data with Allen Cortex label transfer



In [ ]:
DefaultAssay(se_filtered_allen_simp) <- "celltypeprops"

selected_celltypes <- c("Astro", "Endo", "L2/3 IT", "L4", "L5 IT", "L5 PT", "L6 CT", "L6 IT", "L6b", "Lamp5", "Macrophage", "Meis2", "NP", "Oligo", "Peri", "Pvalb", "SMC", "Sncg", "Sst", "VLMC", "Vip")

se_filtered_allen_simp <- LoadImages(se_filtered_allen_simp, verbose = FALSE)

MapFeatures(se_filtered_allen_simp,
            pt_size = 1.5,
            section_number = 2,
            ncol = 5,
            image_use = "transformed",
            features = c("Astro",
                         "L2/3 IT",
                         "L4",
                         "L5 IT",
                         "L5 PT",
                         "L6 CT",
                         "L6 IT",
                         "L6b",
                         "Meis2",
                         "NP",
                         "Oligo",
                         "Pvalb",
                         "Sst", 
                         "VLMC", 
                         "Vip"),
            colors = RColorBrewer::brewer.pal(n = 9, name = "Spectral") |> rev(),
            scale_alpha = TRUE,
            crop_area = c(0.18, 0.48, 0.82, 0.88)) &
  theme(legend.position = "right", legend.margin = margin(b = 50),
        legend.text = element_text(angle = 0),
        plot.title = element_blank())



## Multiple cell types (allen)



In [ ]:
# Plot multiple features
selected_celltypes <- c("Astro", "L2/3 IT", "L4", "L5 IT", "L5 PT", "L6 CT", "L6 IT", "L6b", "Meis2", "NP", "Oligo", "Peri", "Pvalb", "Sncg", "Sst", "VLMC")
MapMultipleFeatures(se_filtered_allen_simp, 
                    image_use = "raw", 
                    pt_size = 1.3, max_cutoff = 0.99,
                    override_plot_dims = TRUE, 
                    colors = c("#114477", "#4477AA", "#77AADD", "#117755", "#44AA88", "#99CCBB", "#777711", "#AAAA44", "#DDDD77", "#771111", "#AA4444", "#DD7777", "#771144", "#AA4477", "#DD77AA"),
                    features = selected_celltypes[1:15]) +
  plot_layout(guides = "collect")



# Zoom in cortex allen deconv



In [ ]:
cols_he <- viridis::viridis(11)

selected_celltypes <- c("L2/3 IT", "L4", "L5 IT", 
                       "L5 PT", "L6 CT", "L6 IT", "L6b", 
                       "Oligo", "Pvalb", "Meis2", "Astro",
                       "VLMC", "SMC")

# Plot multiple features
MapMultipleFeatures(se_filtered_allen_simp,
                    section_number = 2,
                    image_use = "transformed",
                    max_cutoff = 0.99,
                    pt_size = 1.5,
                    colors = c("#332288", "#88CCEE", "#44AA99", "#117733", "#DDCC77", "#CC6677","#AA4499"),
                    features = selected_celltypes[1:7],
                    crop_area = c(0.2, 0.5, 0.8, 0.6)) & theme(plot.title = element_blank())



## Session info


In [ ]:
sessionInfo()